# 01 — Exploratory Data Analysis: CodeNet Dataset

This notebook explores the filtered CodeNet subset and AI-generated solutions.
We examine label distribution, language balance, code length distributions,
and per-problem statistics.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from config.settings import PROCESSED_DIR, SPLITS_DIR

sns.set_theme(style='whitegrid', palette='Set2')
%matplotlib inline

In [ ]:
# Load the full dataset
df = pd.read_parquet(PROCESSED_DIR / 'dataset_full.parquet')
print(f'Total samples: {len(df)}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Label distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['label'].value_counts().plot.bar(ax=axes[0])
axes[0].set_title('Label Distribution')
axes[0].set_xticklabels(['Human (0)', 'AI (1)'], rotation=0)

df.groupby(['language', 'label']).size().unstack().plot.bar(ax=axes[1])
axes[1].set_title('Label Distribution by Language')
axes[1].legend(['Human', 'AI'])
plt.tight_layout()
plt.show()

In [ ]:
# Code length statistics
df['code_length'] = df['code'].str.len()
df['line_count'] = df['code'].str.count('\n') + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, group in df.groupby('label'):
    name = 'Human' if label == 0 else 'AI'
    axes[0].hist(group['code_length'], bins=50, alpha=0.6, label=name)
    axes[1].hist(group['line_count'], bins=50, alpha=0.6, label=name)

axes[0].set_title('Code Length (characters)')
axes[0].legend()
axes[1].set_title('Line Count')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Per-problem sample counts
problem_counts = df.groupby(['problem_id', 'label']).size().unstack(fill_value=0)
problem_counts.columns = ['Human', 'AI']
print(f'Problems: {len(problem_counts)}')
print(problem_counts.describe())

In [ ]:
# AI model distribution
ai_df = df[df['label'] == 1]
if 'model_name' in ai_df.columns:
    model_counts = ai_df['model_name'].value_counts()
    model_counts.plot.bar(figsize=(10, 4), title='AI Solutions by Model')
    plt.tight_layout()
    plt.show()

In [ ]:
# Split sizes
for split_name in ['train', 'val', 'test']:
    split_df = pd.read_parquet(SPLITS_DIR / f'{split_name}.parquet')
    print(f'{split_name}: {len(split_df)} samples, '
          f'{split_df["label"].value_counts().to_dict()}, '
          f'{split_df["problem_id"].nunique()} problems')